<a href="https://colab.research.google.com/github/musfira-tahir-ai/flyrank-assignment-1/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/musfira-tahir-ai/flyrank-assignment-1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

I am framing Lane 2 as a scoring problem.  If i used a standard classification model it would only give us a binary label like "declining" or "healthy." But in the dataset, thousands of mature pages show traffic drops at the same time. So for example, if a system flags 10,000 pages all as "declining," an editorial team still doesn't know where to start. So  i will skip the 2 categorial outputs and instead, by training a scoring model, I get an output of a continuous value for every single page.
The score b/w 0 and 1.0 will then form a prioritized queue.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target Proxy: "Priority Score" based on Traffic Loss and Search Demand
The dataset doesn't have a ready-made column telling us which pages to fix, so I am creating a proxy target using two real, observed numbers from the data:  
Click Loss: How much a page's recent traffic (clicks_30d) has dropped compared to its 90-day average (clicks_90d).  
Audience Size: How many people are still seeing the page on Google (impressions_90d).  

Where it comes from:This target isn't a direct human label. it's a rule built from real numbers. If a page gets thousands of impressions but its clicks are dropping fast, it gets a high score. If a page's traffic drops but nobody is searching for it anyway, it gets a low score. As our key focus here is impressin vs clicks. This creates a clean numerical target that will teach the model what a "high-opportunity fix" looks like.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The metric I can defend best for this business task is Spearman Rank Correlation.
The main objective isn't predicting exact click counts butit's getting the order right so the content team updates the highest-value pages first. Spearman rank correlation will measures how well the model's predicted ranking matches the true opportunity ranking of the pages.

What number means "good"?
A score of 1.0 means perfect ranking order.A score of 0.0 means the ranking is random guessing as predicted and actual had no correlation.For this task, a correlation of $r_s \ge 0.65$ means the model is successfully putting the right pages at the top of the queue.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis: 1 Row = 1 Unique Web Page
In this dataset slice, one row represents one published content page on the site (identified by its unique content_id).  
What the table shows:
dataframe above isolates mature articles which are older than 90 days and brings together their traffic history—age, total search impressions, historical clicks, and recent clicks.
It also displays our calculated target_opportunity_score column right alongside the raw metrics, giving a single-row snapshot for every candidate article.  

In [ ]:
import os
import pandas as pd
import numpy as np

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    !git clone https://github.com/musfira-tahir-ai/flyrank-assignment-1.git
    %cd flyrank-assignment-1

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

#for automatically dentifying correct colomn
possible_click_cols = ["clicks_30d", "clicks_m1", "clicks_last_30d", "clicks_recent"]
click_col = next((c for c in possible_click_cols if c in df.columns), None)

if click_col is None:
    #more error handling
    click_col = [c for c in df.columns if "click" in c.lower() and c != "clicks_90d"][0]

mature_df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()

click_ratio = mature_df[click_col] / (mature_df["clicks_90d"] / 3 + 1e-5)
decay_factor = np.maximum(0, 1 - click_ratio)
mature_df["target_opportunity_score"] = decay_factor * np.log1p(mature_df["impressions_90d"])

display_cols = ["content_id", "content_age_days", "impressions_90d", "clicks_90d", click_col, "trend_direction", "target_opportunity_score"]
print(f"Total Mature Pages (Rows): {len(mature_df):,}\n")
mature_df[display_cols].head(5)

Total Mature Pages (Rows): 30,000



,content_id,content_age_days,impressions_90d,clicks_90d,clicks_last_30d,trend_direction,target_opportunity_score
0,content_304f48230142,187,3803,29,2,down,6.538195
1,content_a1fb4e703a9e,445,15320,7,2,down,1.376747
2,content_9aa793d4d895,141,12581,11,1,down,6.865478
3,content_331d6c4de07b,463,11751,58,22,stable,0.000000
4,content_d99b7a2d90ca,263,19140,24,10,down,0.000000


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

If statement like "flag any page that lost more than 20% of its clicks," but static rules like this will fall apart pretty quickly on real search data for following reasons:

Rules can't weigh impact so a simple rule will treats a popular page losing 2,000 visitors the exact same as a small page losing 2 visitors which doesn't have the same impacts.
so fixing the high-traffic page is a huge win, while fixing the small page barely moves the needle.

Search performance is messy as a page's actual potential depends on a mix of its age, search impressions, current rank, and click trends all at once. Rule will only try to cover every edge case with hardcoded logic which will gets messy fast.  

Rules gives us pile, not an ordered queue so arule just gives me a giant collection of flagged pages with no clear order. Whereas a model gives every page a relative score, so the team can just open the spreadsheet and start from the top.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.